In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.transforms as mtransforms
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from mpl_toolkits.axes_grid1 import make_axes_locatable
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.utils import shuffle
import shap
import joblib
import joblib
from sklearn.base import clone
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter

In [ ]:
# ========================================
# CONFIGURATION
# ========================================
USE_EXTERNAL_VALIDATION = False  # True: run external validation | False: skip validation
SAVE_MODEL = False  # True: save model to .pkl file | False: skip saving
SAVE_CV_RESULTS = True  # Set to False to skip saving
property_name = "rho"           # "Hc", "Js", or "rho"
featurization_method = "WenAlloys"  # "Composition", "WenAlloys", or "Oliynyk"
# K-Fold cross-validation parameters
N_SPLITS = 10
RANDOM_STATE = 73 # rs=42 produced a degenerate fold (low R²); 73 yields balanced splits
SAVE_GRIDSHAP_PLOT = False  # Set to False to skip downloading Grid-SHAP plots
SAVE_BEESWARM_PLOT = False  # Set to False to skip downloading SHAP beeswarm plots

In [ ]:
# Install packages for Google Colab
try:
    import google.colab
    !pip install -q matminer pymatgen
except:
    pass  # Packages already installed locally

In [ ]:
try:
    # For Colab: upload file
    from google.colab import files
    uploaded = files.upload()
    filename = next(iter(uploaded))
    df = pd.read_csv(filename)
except:
    # For local execution: use predefined path
    df = pd.read_csv('../data/training/Dataset_formula.csv')

print(f"Loaded {len(df)} samples with {len(df.columns)} columns")
display(df.head())

In [ ]:
if USE_EXTERNAL_VALIDATION:
    # File creation
    output_file = f"predictions_{property_name}_{featurization_method}.xlsx"
    column_name = f"{property_name}_predict"

    # Load experimental dataset
    try:
        # Colab: upload file
        uploaded = files.upload()
        X_predict = pd.read_csv('wenalloys_for_predict.csv')
    except:
        # Local: load from directory
        X_predict = pd.read_csv('../data/external_validation/wenalloys_for_predict.csv')

    print(f"Loaded {len(X_predict)} experimental samples with {len(X_predict.columns)} features")
    display(X_predict.head())

    # Initialize prediction results table (5 alloys × 2 models)
    results = pd.DataFrame({
        "Model": ["ETR"]*5 + ["XGB"]*5,
        "Alloy_number": list(range(1, 6)) + list(range(1, 6)),
        column_name: [np.nan]*10
    })

    print(f"\nPrediction table initialized for {property_name} ({featurization_method})")
    print(results, "\n")

In [ ]:
# Remove rows with missing target values
df_drop = df.dropna(subset=['Resistivity'])
# Select formula and target columns for featurization
df_final = df_drop.loc[:, ['formula', 'Resistivity']]

In [ ]:
%%capture

# Convert formula strings to Composition objects
from matminer.featurizers.conversions import StrToComposition

str_comp = StrToComposition(target_col_id='composition')
df_conversion = str_comp.featurize_dataframe(df_final, col_id='formula')

In [ ]:
# ====================================================================
# LOAD wenalloys_corrected.py FROM GITHUB
# ====================================================================
import os
import urllib.request

# URL to raw file on GitHub
WENALLOYS_URL = "https://raw.githubusercontent.com/Niccuby/Sendust-ml-featurization-SHAP/main/scripts/wenalloys_corrected.py"

# Download if not exists
if not os.path.exists('wenalloys_corrected.py'):
    print("Downloading wenalloys_corrected.py from GitHub...")
    urllib.request.urlretrieve(WENALLOYS_URL, 'wenalloys_corrected.py')
    print("✓ Downloaded successfully")
else:
    print("✓ wenalloys_corrected.py already exists")

# Import module
from wenalloys_corrected import CorrectedWenAlloys

In [ ]:
%%capture

# Initialize corrected featurizer
featurizer = CorrectedWenAlloys()

# Extract WenAlloys features from composition objects
data_featurized = featurizer.featurize_dataframe(df_conversion, col_id='composition')

# Extract target variable
y = data_featurized['Resistivity']

In [ ]:
# Select only WenAlloys features (exclude formula, composition, and other metadata columns)
data_features_only = data_featurized.iloc[:, 5:]

# Remove low-variance features (keep only features with 4+ unique values)
df_drop_unique = data_features_only.loc[:, (data_features_only.nunique() >= 4)]

In [ ]:
def select_wenalloys_features(df_drop_unique: pd.DataFrame) -> pd.DataFrame:
    """
    Extract manually selected WenAlloys features for Sendust alloy modeling.

    Args:
        df_drop_unique: DataFrame with all WenAlloys features after variance filtering

    Returns:
        DataFrame with 11 manually selected physically meaningful features
    """
    # List of 11 manually selected features based on domain knowledge
    selected_features = [
        'Yang delta',
        'Yang omega',
        'APE mean',
        'Configuration entropy',
        'Atomic weight mean',
        'Electronegativity delta',
        'VEC mean',
        'Mixing enthalpy',
        'Mean cohesive energy',
        'Shear modulus strength model',
        'Lambda entropy'
    ]

    # Check which features are available in the dataframe
    available_features = [feature for feature in selected_features if feature in df_drop_unique.columns]

    # Warn if any features are missing
    if len(available_features) != len(selected_features):
        missing_features = set(selected_features) - set(available_features)
        print(f"Warning: Following features not found and will be skipped: {list(missing_features)}")

    # Return dataframe with selected features only
    return df_drop_unique[available_features].copy()


# Apply manual feature selection (11 features for WenAlloys)
df_the_features = select_wenalloys_features(df_drop_unique)

In [ ]:
# Rename features to compact scientific notation for better readability
df_renamed = df_the_features.rename(columns={
    'Yang delta': 'δ',
    'Atomic weight mean': 'A̅',
    'Electronegativity delta': 'δχ',
    'Mixing enthalpy': 'ΔHmix',
    'Lambda entropy': 'Λ',
    'Configuration entropy': 'ΔSmix',
    'VEC mean': 'VEC',
    'Shear modulus strength model': 'G'
})

In [ ]:
# Prepare feature matrix and target vector
features = df_renamed
target = y

# Shuffle data to ensure random distribution
features, target = shuffle(features, target, random_state=42)

# Split into training (80%) and test (20%) sets
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")

In [ ]:
# ============================================================
# LOAD FOLDS (generated in Composition notebook)
# ============================================================

features_cv = features.reset_index(drop=True)
target_cv   = target.reset_index(drop=True)

import os

train_file = f'folds_{property_name}_train.npy'
test_file  = f'folds_{property_name}_test.npy'

if not os.path.exists(train_file) or not os.path.exists(test_file):
    try:
        # Colab: manual upload
        from google.colab import files
        uploaded = files.upload()
    except:
        # Fallback: download from GitHub
        import urllib.request
        base_url = "https://github.com/Niccuby/Sendust-ml-featurization-SHAP/raw/refs/heads/main/data/splits"
        urllib.request.urlretrieve(f"{base_url}/{train_file}", train_file)
        urllib.request.urlretrieve(f"{base_url}/{test_file}",  test_file)
        print(f"✅ Folds downloaded from GitHub")
else:
    print(f"✅ Folds found locally, skipping download")

fold_train = np.load(train_file, allow_pickle=True)
fold_test  = np.load(test_file,  allow_pickle=True)
folds      = list(zip(fold_train, fold_test))

print(f"Folds loaded: {len(folds)} | Total samples: {sum(len(te) for _, te in folds)}")

# Prediction Properties

In [ ]:
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
import xgboost as xgb

# Random Forest

In [ ]:
# Train Random Forest model with pre-optimized hyperparameters
best_params = {
        'n_estimators': 460,
        'max_depth': 10,
        'min_samples_split': 2,
        'min_samples_leaf': 1,
        'criterion': 'absolute_error'
    }
best_model = RandomForestRegressor(random_state=42, **best_params)
best_model.fit(X_train, y_train)

In [ ]:
# ============================================================
# FINAL MODEL + CROSS-VALIDATION
# ============================================================

cv_all_results = {}

final_model = RandomForestRegressor(random_state=42, **best_params)
final_model.fit(features_cv, target_cv)

fold_results = []
all_y_true   = []
all_y_pred   = []
fold_indices = []

print(f"\n{'='*65}")
print(f"CV | Target: {property_name} | Features: {featurization_method} | N={len(target_cv)}")
print(f"{'='*65}")
print(f"{'Fold':<6} {'Train':<8} {'Test':<7} {'R²_train':<12} {'R²_test':<12} {'MAE_test'}")
print(f"{'-'*65}")

for fold, (train_idx, test_idx) in enumerate(folds, start=1):
    X_train_f = features_cv.iloc[train_idx]
    X_test_f  = features_cv.iloc[test_idx]
    y_train_f = target_cv.iloc[train_idx]
    y_test_f  = target_cv.iloc[test_idx]

    model_f = clone(final_model)
    model_f.fit(X_train_f, y_train_f)

    y_pred_train = model_f.predict(X_train_f)
    y_pred_test  = model_f.predict(X_test_f)

    r2_tr  = r2_score(y_train_f, y_pred_train)
    r2_te  = r2_score(y_test_f,  y_pred_test)
    mae_te = mean_absolute_error(y_test_f, y_pred_test)

    fold_results.append({
        'Fold': fold, 'train_size': len(train_idx), 'test_size': len(test_idx),
        'R2_train': r2_tr, 'R2_test': r2_te, 'MAE_test': mae_te,
        'test_idx': test_idx
    })

    all_y_true.extend(y_test_f.values)
    all_y_pred.extend(y_pred_test)
    fold_indices.extend([fold] * len(test_idx))

    print(f"{fold:<6} {len(train_idx):<8} {len(test_idx):<7} "
          f"{r2_tr:<12.4f} {r2_te:<12.4f} {mae_te:.4f}")

cv_all_results['RFR'] = (fold_results, all_y_true, all_y_pred)

print(f"{'-'*65}")
r2_te_vals  = [r['R2_test']  for r in fold_results]
mae_te_vals = [r['MAE_test'] for r in fold_results]
print(f"{'':6} {'':8} {'':7} {'Mean':<12} {np.mean(r2_te_vals):<12.4f} {np.mean(mae_te_vals):.4f}")
print(f"{'':6} {'':8} {'':7} {'±Std':<12} {np.std(r2_te_vals):<12.4f} {np.std(mae_te_vals):.4f}")

oof_r2 = r2_score(all_y_true, all_y_pred)
print(f"{'':6} {'':8} {'':7} {'OOF R²':<12} {oof_r2:.4f}")
print(f"{'='*65}")

In [ ]:
if SAVE_MODEL:
    # Save trained model
    joblib.dump(final_model, '1_wenalloys_resistivity_rfr.pkl')
    print("✅ Model saved as '1_wenalloys_resistivity_rfr.pkl'")

    try:
        files.download('1_wenalloys_resistivity_rfr.pkl')
    except:
        pass

# Extra Trees Regressor

In [ ]:
# Train Extra Trees model with pre-optimized hyperparameters
best_params = {
        'n_estimators': 304,
        'max_depth': 8,
        'min_samples_split': 2,
        'min_samples_leaf': 1,
        'criterion': 'absolute_error'
    }

best_model = ExtraTreesRegressor(random_state=42, **best_params)
best_model.fit(X_train, y_train)

In [ ]:
# ============================================================
# FINAL MODEL + CROSS-VALIDATION
# ============================================================

final_model = ExtraTreesRegressor(random_state=42, **best_params)
final_model.fit(features_cv, target_cv)

fold_results = []
all_y_true   = []
all_y_pred   = []
fold_indices = []

print(f"\n{'='*65}")
print(f"CV | Target: {property_name} | Features: {featurization_method} | N={len(target_cv)}")
print(f"{'='*65}")
print(f"{'Fold':<6} {'Train':<8} {'Test':<7} {'R²_train':<12} {'R²_test':<12} {'MAE_test'}")
print(f"{'-'*65}")

for fold, (train_idx, test_idx) in enumerate(folds, start=1):
    X_train_f = features_cv.iloc[train_idx]
    X_test_f  = features_cv.iloc[test_idx]
    y_train_f = target_cv.iloc[train_idx]
    y_test_f  = target_cv.iloc[test_idx]

    model_f = clone(final_model)
    model_f.fit(X_train_f, y_train_f)

    y_pred_train = model_f.predict(X_train_f)
    y_pred_test  = model_f.predict(X_test_f)

    r2_tr  = r2_score(y_train_f, y_pred_train)
    r2_te  = r2_score(y_test_f,  y_pred_test)
    mae_te = mean_absolute_error(y_test_f, y_pred_test)

    fold_results.append({
        'Fold': fold, 'train_size': len(train_idx), 'test_size': len(test_idx),
        'R2_train': r2_tr, 'R2_test': r2_te, 'MAE_test': mae_te,
        'test_idx': test_idx
    })

    all_y_true.extend(y_test_f.values)
    all_y_pred.extend(y_pred_test)
    fold_indices.extend([fold] * len(test_idx))

    print(f"{fold:<6} {len(train_idx):<8} {len(test_idx):<7} "
          f"{r2_tr:<12.4f} {r2_te:<12.4f} {mae_te:.4f}")

cv_all_results['ETR'] = (fold_results, all_y_true, all_y_pred)

print(f"{'-'*65}")
r2_te_vals  = [r['R2_test']  for r in fold_results]
mae_te_vals = [r['MAE_test'] for r in fold_results]
print(f"{'':6} {'':8} {'':7} {'Mean':<12} {np.mean(r2_te_vals):<12.4f} {np.mean(mae_te_vals):.4f}")
print(f"{'':6} {'':8} {'':7} {'±Std':<12} {np.std(r2_te_vals):<12.4f} {np.std(mae_te_vals):.4f}")

oof_r2 = r2_score(all_y_true, all_y_pred)
print(f"{'':6} {'':8} {'':7} {'OOF R²':<12} {oof_r2:.4f}")
print(f"{'='*65}")

In [ ]:
if SAVE_MODEL:

# Save model to .pkl file
    joblib.dump(final_model, '2_wenalloys_resistivity_etr.pkl')

    print("✅ Model saved as '2_wenalloys_resistivity_etr.pkl'")
    try:
        files.download('2_wenalloys_resistivity_etr.pkl')
    except:
        pass

In [ ]:
if USE_EXTERNAL_VALIDATION:
    # Make predictions on external validation set (ETR)
    y_pred_etr = final_model.predict(X_predict)

    # Save ETR predictions to results DataFrame
    results.loc[results["Model"] == "ETR", column_name] = y_pred_etr
    print(f"ETR predictions saved: {y_pred_etr}")
    print(results)
    print("\n")

In [ ]:
# ====================================================================
# SHAP BEESWARM VISUALIZATION
# ====================================================================

explainer   = shap.TreeExplainer(final_model, features_cv)
shap_values = explainer(features_cv)

mpl.rcParams.update({
    'font.size': 23,
    'axes.labelsize': 23,
    'xtick.labelsize': 23,
    'ytick.labelsize': 23,
})

fig, ax = plt.subplots(figsize=(8, 6))

shap.plots.beeswarm(
    shap_values,
    max_display=5,
    group_remaining_features=False,
    show=False,
    s=150,
    ax=ax,
    plot_size=None
)

ax.set_xlabel("SHAP Values", fontsize=23)
ax.tick_params(axis='x', labelsize=23, pad=2)
ax.tick_params(axis='y', which='both', length=0, pad=1, labelsize=23)

dx_pt = -15
offset = mtransforms.ScaledTranslation(dx_pt/72.0, 0, fig.dpi_scale_trans)
for t in ax.get_yticklabels():
    t.set_ha('right')
    t.set_transform(t.get_transform() + offset)

ax.spines['left'].set_position(('axes', 0.0))
ax.yaxis.set_ticks_position('left')

cax = fig.axes[-1]
cax.tick_params(pad=2, labelsize=23)
cax.set_ylabel("Feature Value", fontsize=23, labelpad=4)
cax.yaxis.set_label_coords(2.5, 0.5)

fig.tight_layout(pad=0.2)
beeswarm_filename = f'{featurization_method}_SHAP_beeswarm_{property_name}_ETR.png'
fig.savefig(beeswarm_filename, dpi=600, bbox_inches='tight', transparent=False)
plt.show()

if SAVE_BEESWARM_PLOT:
    try:
        files.download(beeswarm_filename)
    except:
        pass

In [ ]:
def shifted_coolwarm_cmap(p, n=256):
    """Shift coolwarm colormap so position p becomes neutral center."""
    base = plt.get_cmap("coolwarm")
    xs = np.linspace(0, 1, n)

    cols = []
    for x in xs:
        if x <= p:
            t = 0.5 * (x / p) if p > 0 else 0.0
        else:
            t = 0.5 + 0.5 * ((x - p) / (1 - p)) if p < 1 else 1.0
        cols.append(base(t))

    return mcolors.ListedColormap(cols, name="coolwarm_shifted")


def make_coolwarm_saturated_with_center(values, vcenter=0.0, nbins=6, steps=(1, 2, 2.5, 5, 10)):
    """Create norm, cmap, and ticks for SHAP with saturated colors and zero at neutral."""
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]

    # Fallback for empty data
    if x.size == 0:
        norm = mcolors.Normalize(vmin=-1, vmax=1, clip=True)
        cmap = plt.get_cmap("coolwarm")
        ticks = np.array([-1, 0, 1], dtype=float)
        return norm, cmap, ticks

    data_min, data_max = float(np.min(x)), float(np.max(x))

    # Normalize by actual data range for saturated colors
    norm = mcolors.Normalize(vmin=data_min, vmax=data_max, clip=True)

    # Generate ticks with scientific steps
    loc = mticker.MaxNLocator(nbins=nbins, steps=list(steps))
    ticks = loc.tick_values(data_min, data_max)
    ticks = ticks[(ticks >= norm.vmin) & (ticks <= norm.vmax)]

    # Shift colormap to place vcenter at neutral
    if data_min < vcenter < data_max:
        p = (vcenter - data_min) / (data_max - data_min)
        cmap = shifted_coolwarm_cmap(p, n=256)

        # Ensure vcenter is in ticks
        if not np.any(np.isclose(ticks, vcenter)):
            ticks = np.sort(np.append(ticks, vcenter))
    else:
        cmap = plt.get_cmap("coolwarm")

    return norm, cmap, ticks

In [ ]:
# ====================================================================
# GRID-SHAP VISUALIZATION
# ====================================================================

explainer   = shap.TreeExplainer(final_model, features_cv)
shap_values = explainer(features_cv)

mean_abs_shap_values = np.abs(shap_values.values).mean(axis=0)
top5_indices = np.argsort(mean_abs_shap_values)[-5:][::-1]

plt.style.use('default')
fig = plt.figure(figsize=(8, 10))
gs = GridSpec(3, 2, figure=fig, wspace=0.65, hspace=0.5)

for i, feature_index in enumerate(top5_indices):
    feature_name = features_cv.columns[feature_index]
    row = i // 2
    col = i % 2

    ax = fig.add_subplot(gs[row, col])
    ax.set_facecolor('#f0f0f0')

    shap_col = shap_values[:, feature_index].values

    norm, cmap_local, ticks = make_coolwarm_saturated_with_center(
        shap_col,
        vcenter=0.0,
        nbins=6
    )

    sc = ax.scatter(
        features_cv[feature_name],
        target_cv,
        c=shap_col,
        cmap=cmap_local,
        norm=norm,
        alpha=0.9,
        s=40,
        edgecolor='black',
        linewidth=0.25,
        marker='o'
    )

    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='5%', pad=0.05)
    cbar = fig.colorbar(sc, cax=cax)

    cbar.set_ticks(ticks)
    cbar.formatter = mticker.ScalarFormatter(useMathText=False)
    cbar.formatter.set_powerlimits((-2, 3))
    cbar.update_ticks()
    cbar.ax.tick_params(labelsize=11)
    cbar.set_label('SHAP Value', fontsize=12, labelpad=5)

    ax.yaxis.set_major_locator(mticker.MaxNLocator(min_n_ticks=4, nbins=5))
    ax.xaxis.set_major_locator(mticker.MaxNLocator(min_n_ticks=3, nbins=4, integer=True))
    ax.set_xlabel(feature_name, fontsize=12, labelpad=4)
    ax.set_ylabel("Resistivity (μΩ⋅cm)", fontsize=12, labelpad=4)
    ax.tick_params(axis='both', labelsize=11, pad=4)
    ax.grid(True, linestyle=':', alpha=0.35, linewidth=0.45)

gridshap_filename = f'{featurization_method}_Grid-SHAP_{property_name}_ETR.png'
plt.savefig(gridshap_filename, dpi=600, bbox_inches='tight', facecolor='white')
plt.show()

if SAVE_GRIDSHAP_PLOT:
    try:
        files.download(gridshap_filename)
    except:
        pass

# Gradient Boost Regressor

In [ ]:
# Train Gradient Boosting model with pre-optimized hyperparameters
best_params = {
        'n_estimators': 442,
        'learning_rate': 0.04,
        'max_depth': 3,
        'min_samples_split': 17,
        'subsample': 0.8
    }
best_model = GradientBoostingRegressor(random_state=42, **best_params)
best_model.fit(X_train, y_train)

In [ ]:
# ============================================================
# FINAL MODEL + CROSS-VALIDATION
# ============================================================

final_model = GradientBoostingRegressor(random_state=42, **best_params)
final_model.fit(features_cv, target_cv)

fold_results = []
all_y_true   = []
all_y_pred   = []
fold_indices = []

print(f"\n{'='*65}")
print(f"CV | Target: {property_name} | Features: {featurization_method} | N={len(target_cv)}")
print(f"{'='*65}")
print(f"{'Fold':<6} {'Train':<8} {'Test':<7} {'R²_train':<12} {'R²_test':<12} {'MAE_test'}")
print(f"{'-'*65}")

for fold, (train_idx, test_idx) in enumerate(folds, start=1):
    X_train_f = features_cv.iloc[train_idx]
    X_test_f  = features_cv.iloc[test_idx]
    y_train_f = target_cv.iloc[train_idx]
    y_test_f  = target_cv.iloc[test_idx]

    model_f = clone(final_model)
    model_f.fit(X_train_f, y_train_f)

    y_pred_train = model_f.predict(X_train_f)
    y_pred_test  = model_f.predict(X_test_f)

    r2_tr  = r2_score(y_train_f, y_pred_train)
    r2_te  = r2_score(y_test_f,  y_pred_test)
    mae_te = mean_absolute_error(y_test_f, y_pred_test)

    fold_results.append({
        'Fold': fold, 'train_size': len(train_idx), 'test_size': len(test_idx),
        'R2_train': r2_tr, 'R2_test': r2_te, 'MAE_test': mae_te,
        'test_idx': test_idx
    })

    all_y_true.extend(y_test_f.values)
    all_y_pred.extend(y_pred_test)
    fold_indices.extend([fold] * len(test_idx))

    print(f"{fold:<6} {len(train_idx):<8} {len(test_idx):<7} "
          f"{r2_tr:<12.4f} {r2_te:<12.4f} {mae_te:.4f}")

cv_all_results['GBR'] = (fold_results, all_y_true, all_y_pred)

print(f"{'-'*65}")
r2_te_vals  = [r['R2_test']  for r in fold_results]
mae_te_vals = [r['MAE_test'] for r in fold_results]
print(f"{'':6} {'':8} {'':7} {'Mean':<12} {np.mean(r2_te_vals):<12.4f} {np.mean(mae_te_vals):.4f}")
print(f"{'':6} {'':8} {'':7} {'±Std':<12} {np.std(r2_te_vals):<12.4f} {np.std(mae_te_vals):.4f}")

oof_r2 = r2_score(all_y_true, all_y_pred)
print(f"{'':6} {'':8} {'':7} {'OOF R²':<12} {oof_r2:.4f}")
print(f"{'='*65}")

In [ ]:
if SAVE_MODEL:

# Save model to .pkl file
    joblib.dump(final_model, '3_wenalloys_resistivity_gbr.pkl')

    print("✅ Model saved as '3_wenalloys_resistivity_gbr.pkl'")
    try:
        files.download('3_wenalloys_resistivity_gbr.pkl')
    except:
        pass

# XGBoost Regressor

In [ ]:
# Train XGBoost model with pre-optimized hyperparameters
best_params = {
        'objective': 'reg:squarederror',
        'n_estimators': 461,
        'max_depth': 6,
        'learning_rate': 0.2345269441539146,
        'subsample': 0.6524034514469835,
        'colsample_bytree': 0.9817378593570328,
        'min_child_weight': 2,
        'gamma': 0.2998781940594144,
        'alpha': 5.422366535485192,
        'reg_lambda': 9.3158705998517
    }
mean_target = float(y_train.mean())
best_model = xgb.XGBRegressor(random_state=42, **best_params)
best_model.fit(X_train, y_train)

In [ ]:
# ============================================================
# FINAL MODEL + CROSS-VALIDATION
# ============================================================

final_model = xgb.XGBRegressor(random_state=42, **best_params)
final_model.fit(features_cv, target_cv)

fold_results = []
all_y_true   = []
all_y_pred   = []
fold_indices = []

print(f"\n{'='*65}")
print(f"CV | Target: {property_name} | Features: {featurization_method} | N={len(target_cv)}")
print(f"{'='*65}")
print(f"{'Fold':<6} {'Train':<8} {'Test':<7} {'R²_train':<12} {'R²_test':<12} {'MAE_test'}")
print(f"{'-'*65}")

for fold, (train_idx, test_idx) in enumerate(folds, start=1):
    X_train_f = features_cv.iloc[train_idx]
    X_test_f  = features_cv.iloc[test_idx]
    y_train_f = target_cv.iloc[train_idx]
    y_test_f  = target_cv.iloc[test_idx]

    model_f = clone(final_model)
    model_f.fit(X_train_f, y_train_f)

    y_pred_train = model_f.predict(X_train_f)
    y_pred_test  = model_f.predict(X_test_f)

    r2_tr  = r2_score(y_train_f, y_pred_train)
    r2_te  = r2_score(y_test_f,  y_pred_test)
    mae_te = mean_absolute_error(y_test_f, y_pred_test)

    fold_results.append({
        'Fold': fold, 'train_size': len(train_idx), 'test_size': len(test_idx),
        'R2_train': r2_tr, 'R2_test': r2_te, 'MAE_test': mae_te,
        'test_idx': test_idx
    })

    all_y_true.extend(y_test_f.values)
    all_y_pred.extend(y_pred_test)
    fold_indices.extend([fold] * len(test_idx))

    print(f"{fold:<6} {len(train_idx):<8} {len(test_idx):<7} "
          f"{r2_tr:<12.4f} {r2_te:<12.4f} {mae_te:.4f}")

cv_all_results['XGB'] = (fold_results, all_y_true, all_y_pred)

print(f"{'-'*65}")
r2_te_vals  = [r['R2_test']  for r in fold_results]
mae_te_vals = [r['MAE_test'] for r in fold_results]
print(f"{'':6} {'':8} {'':7} {'Mean':<12} {np.mean(r2_te_vals):<12.4f} {np.mean(mae_te_vals):.4f}")
print(f"{'':6} {'':8} {'':7} {'±Std':<12} {np.std(r2_te_vals):<12.4f} {np.std(mae_te_vals):.4f}")

oof_r2 = r2_score(all_y_true, all_y_pred)
print(f"{'':6} {'':8} {'':7} {'OOF R²':<12} {oof_r2:.4f}")
print(f"{'='*65}")

In [ ]:
if SAVE_MODEL:

# Save model to .pkl file
    joblib.dump(final_model, '4_wenalloys_resistivity_xgb.pkl')

    print("✅ Model saved as '4_wenalloys_resistivity_xgb.pkl'")
    try:
        files.download('4_wenalloys_resistivity_xgb.pkl')
    except:
        pass

In [ ]:
if USE_EXTERNAL_VALIDATION:
    # Make predictions on external validation set
    y_pred_xgb = final_model.predict(X_predict)

    # Save predictions to results DataFrame
    results.loc[results["Model"] == "XGB", column_name] = y_pred_xgb
    print(f"XGB predictions saved: {y_pred_xgb}")
    print(results)
    print("\n")

    # Add experimental ground truth values
    y_true = [86.8, 106.9, 146.1, 122.8, 101.2]  # ← UPDATE WITH ACTUAL VALUES
    results[f"{property_name}_true"] = y_true * 2  # Duplicate for both models

    # Calculate per-sample errors
    results["Residual"]  = results[column_name] - results[f"{property_name}_true"]
    results["AE"]        = results["Residual"].abs()                                      # Absolute Error
    results["APE (%)"]   = 100 * results["AE"] / results[f"{property_name}_true"]        # Absolute Percentage Error

    # Calculate average metrics per model (MAE and MAPE)
    for model in ["ETR", "XGB"]:
        mask = results["Model"] == model
        mae  = results.loc[mask, "AE"].mean()
        mape = results.loc[mask, "APE (%)"].mean()
        print(f"{model}: MAE = {mae:.4f}, MAPE = {mape:.2f}%")

    # Save results to Excel
    results.to_excel(output_file, index=False, engine='openpyxl')
    print(f"✓ Results saved to {output_file}")

    # Download file (Colab only)
    try:
        files.download(output_file)
    except:
        pass

In [ ]:
# ====================================================================
# SHAP BEESWARM VISUALIZATION (XGBOOST)
# ====================================================================

dmatrix_cv = xgb.DMatrix(features_cv, feature_names=features_cv.columns.tolist())

shap_values_raw    = final_model.get_booster().predict(dmatrix_cv, pred_contribs=True)
shap_values_matrix = shap_values_raw[:, :-1]
expected_value     = float(shap_values_raw[0, -1])

print(f"Shape SHAP values: {shap_values_matrix.shape}")
print(f"Expected value: {expected_value}")

explanation = shap.Explanation(
    values=shap_values_matrix,
    base_values=np.full(len(features_cv), expected_value),
    data=features_cv.values,
    feature_names=features_cv.columns.tolist()
)

mpl.rcParams.update({
    'font.size': 23,
    'axes.labelsize': 23,
    'xtick.labelsize': 23,
    'ytick.labelsize': 23,
})

fig, ax = plt.subplots(figsize=(8, 6))

shap.plots.beeswarm(
    explanation,
    max_display=5,
    group_remaining_features=False,
    show=False,
    s=150,
    ax=ax,
    plot_size=None
)

ax.set_xlabel("SHAP Values", fontsize=23)
ax.tick_params(axis='x', labelsize=23, pad=2)
ax.tick_params(axis='y', which='both', length=0, pad=1, labelsize=23)

desired_ticks = [-40, -20, 0, 20, 40]
ax.set_xticks(desired_ticks)

dx_pt = -3
offset = mtransforms.ScaledTranslation(dx_pt/72.0, 0, fig.dpi_scale_trans)
for t in ax.get_yticklabels():
    t.set_ha('right')
    t.set_transform(t.get_transform() + offset)

ax.spines['left'].set_position(('axes', 0.0))
ax.yaxis.set_ticks_position('left')

cax = fig.axes[-1]
cax.tick_params(pad=2, labelsize=23)
cax.set_ylabel("Feature Value", fontsize=23, labelpad=4)
cax.yaxis.set_label_coords(2.5, 0.5)

fig.tight_layout(pad=0.2)
beeswarm_filename = f'{featurization_method}_SHAP_beeswarm_{property_name}_XGB.png'
fig.savefig(beeswarm_filename, dpi=600, bbox_inches='tight', transparent=False)
plt.show()

if SAVE_BEESWARM_PLOT:
    try:
        files.download(beeswarm_filename)
    except:
        pass

In [ ]:
# ====================================================================
# CUSTOM COLORMAP FUNCTIONS
# ====================================================================

def shifted_coolwarm_cmap(p, n=256):
    """Shift coolwarm colormap so position p becomes neutral center."""
    base = plt.get_cmap("coolwarm")
    xs = np.linspace(0, 1, n)

    cols = []
    for x in xs:
        if x <= p:
            t = 0.5 * (x / p) if p > 0 else 0.0
        else:
            t = 0.5 + 0.5 * ((x - p) / (1 - p)) if p < 1 else 1.0
        cols.append(base(t))

    return mcolors.ListedColormap(cols, name="coolwarm_shifted")


def make_coolwarm_saturated_with_center(values, vcenter=0.0, nbins=6, steps=(1, 2, 2.5, 5, 10)):
    """Create norm, cmap, and ticks for SHAP with saturated colors and zero at neutral."""
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]

    # Fallback for empty data
    if x.size == 0:
        norm = mcolors.Normalize(vmin=-1, vmax=1, clip=True)
        cmap = plt.get_cmap("coolwarm")
        ticks = np.array([-1, 0, 1], dtype=float)
        return norm, cmap, ticks

    data_min, data_max = float(np.min(x)), float(np.max(x))

    # Normalize by actual data range for saturated colors
    norm = mcolors.Normalize(vmin=data_min, vmax=data_max, clip=True)

    # Generate ticks with scientific steps
    loc = mticker.MaxNLocator(nbins=nbins, steps=list(steps))
    ticks = loc.tick_values(data_min, data_max)
    ticks = ticks[(ticks >= norm.vmin) & (ticks <= norm.vmax)]

    # Shift colormap to place vcenter at neutral
    if data_min < vcenter < data_max:
        p = (vcenter - data_min) / (data_max - data_min)
        cmap = shifted_coolwarm_cmap(p, n=256)

        # Ensure vcenter is in ticks
        if not np.any(np.isclose(ticks, vcenter)):
            ticks = np.sort(np.append(ticks, vcenter))
    else:
        cmap = plt.get_cmap("coolwarm")

    return norm, cmap, ticks


In [ ]:
# ====================================================================
# GRID-SHAP VISUALIZATION FOR XGBOOST
# ====================================================================

dmatrix_cv = xgb.DMatrix(features_cv, feature_names=features_cv.columns.tolist())

shap_values_raw    = final_model.get_booster().predict(dmatrix_cv, pred_contribs=True)
shap_values_matrix = shap_values_raw[:, :-1]
expected_value     = float(shap_values_raw[0, -1])

explanation = shap.Explanation(
    values=shap_values_matrix,
    base_values=np.full(len(features_cv), expected_value),
    data=features_cv.values,
    feature_names=features_cv.columns.tolist()
)

print(f"Shape SHAP values: {shap_values_matrix.shape}")
print(f"Expected value: {expected_value}")

mean_abs_shap_values = np.abs(explanation.values).mean(axis=0)
top5_indices = np.argsort(mean_abs_shap_values)[-5:][::-1]

plt.style.use('default')
fig = plt.figure(figsize=(8, 10))
gs = GridSpec(3, 2, figure=fig, wspace=0.7, hspace=0.5)

for i, feature_index in enumerate(top5_indices):
    feature_name = features_cv.columns[feature_index]
    row = i // 2
    col = i % 2

    ax = fig.add_subplot(gs[row, col])
    ax.set_facecolor('#f0f0f0')

    shap_col = explanation[:, feature_index].values

    norm, cmap_local, ticks = make_coolwarm_saturated_with_center(
        shap_col,
        vcenter=0.0,
        nbins=6
    )

    sc = ax.scatter(
        features_cv[feature_name],
        target_cv,
        c=shap_col,
        cmap=cmap_local,
        norm=norm,
        alpha=0.9,
        s=40,
        edgecolor='black',
        linewidth=0.25,
        marker='o'
    )

    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='5%', pad=0.05)
    cbar = fig.colorbar(sc, cax=cax)

    cbar.set_ticks(ticks)
    cbar.formatter = mticker.ScalarFormatter(useMathText=False)
    cbar.formatter.set_powerlimits((-2, 3))
    cbar.update_ticks()
    cbar.ax.tick_params(labelsize=11)
    cbar.set_label('SHAP Value', fontsize=12, labelpad=5)

    ax.yaxis.set_major_locator(mticker.MaxNLocator(min_n_ticks=4, nbins=5))
    ax.xaxis.set_major_locator(mticker.MaxNLocator(min_n_ticks=3, nbins=4, integer=True))
    ax.set_xlabel(feature_name, fontsize=12, labelpad=4)
    ax.set_ylabel("Resistivity (μΩ⋅cm)", fontsize=12, labelpad=4)
    ax.tick_params(axis='both', labelsize=11, pad=4)
    ax.grid(True, linestyle=':', alpha=0.35, linewidth=0.45)

gridshap_filename = f'{featurization_method}_Grid-SHAP_{property_name}_XGB.png'
plt.savefig(gridshap_filename, dpi=600, bbox_inches='tight', facecolor='white')
plt.show()

if SAVE_GRIDSHAP_PLOT:
    try:
        files.download(gridshap_filename)
    except:
        pass

# **Results**

In [ ]:
def build_cv_dataframe(fold_results, all_y_true, all_y_pred):
    """Convert fold_results list into a DataFrame with Mean, Std and OOF R² rows."""
    rows = []

    for r in fold_results:
        rows.append({
            'Fold':       r['Fold'],
            'Train size': r['train_size'],
            'Test size':  r['test_size'],
            'R² train':   round(r['R2_train'], 4),
            'R² test':    round(r['R2_test'],  4),
            'MAE test':   round(r['MAE_test'],  4),
        })

    r2_tr_vals  = [r['R2_train'] for r in fold_results]
    r2_te_vals  = [r['R2_test']  for r in fold_results]
    mae_te_vals = [r['MAE_test'] for r in fold_results]

    rows.append({
        'Fold':       '',
        'Train size': '',
        'Test size':  '',
        'R² train':   'Mean',
        'R² test':    round(np.mean(r2_te_vals),  4),
        'MAE test':   round(np.mean(mae_te_vals), 4),
    })

    rows.append({
        'Fold':       '',
        'Train size': '',
        'Test size':  '',
        'R² train':   '±Std',
        'R² test':    round(np.std(r2_te_vals),  4),
        'MAE test':   round(np.std(mae_te_vals), 4),
    })

    # OOF R² row
    rows.append({
        'Fold':       '',
        'Train size': '',
        'Test size':  '',
        'R² train':   'OOF R²',
        'R² test':    round(r2_score(all_y_true, all_y_pred), 4),
        'MAE test':   '',
    })

    return pd.DataFrame(rows)


def save_cv_to_xlsx(cv_all_results, property_name, featurization_method, filename=None):
    """Save CV results for all models to a single .xlsx with one sheet per model."""
    if filename is None:
        filename = f'cv_results_{property_name}_{featurization_method}.xlsx'

    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        for model_name, (fold_results, all_y_true, all_y_pred) in cv_all_results.items():
            df = build_cv_dataframe(fold_results, all_y_true, all_y_pred)
            df.to_excel(writer, sheet_name=model_name, index=False)

    wb = load_workbook(filename)
    header_fill = PatternFill('solid', fgColor='2E4057')
    mean_fill   = PatternFill('solid', fgColor='D9E8F5')
    std_fill    = PatternFill('solid', fgColor='EAF4FB')
    oof_fill    = PatternFill('solid', fgColor='D6EAD6')

    header_font = Font(bold=True, color='FFFFFF')
    bold_font   = Font(bold=True)
    center_align = Alignment(horizontal='center')

    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]

        # Auto column width
        for col in ws.columns:
            max_len = max(len(str(cell.value)) if cell.value else 0 for cell in col)
            ws.column_dimensions[get_column_letter(col[0].column)].width = max_len + 4

        for row in ws.iter_rows():
            for cell in row:
                cell.alignment = center_align


                if cell.row == 1:
                    cell.fill = header_fill
                    cell.font = header_font
                    continue


                row_label = str(ws.cell(row=cell.row, column=4).value)

                if row_label in ['Mean', '±Std', 'OOF R²']:

                    if cell.column >= 4:
                        if row_label == 'Mean':
                            cell.fill = mean_fill
                            if cell.column in [4, 5, 6]: cell.font = bold_font
                        elif row_label == '±Std':
                            cell.fill = std_fill
                        elif row_label == 'OOF R²':
                            cell.fill = oof_fill
                            if cell.column in [4, 5, 6]: cell.font = bold_font

        # Add sheet title above the table
        ws.insert_rows(1)
        ws.cell(row=1, column=1,
                value=f'{sheet_name} | Property: {property_name} | Features: {featurization_method}')
        ws.cell(row=1, column=1).font = Font(bold=True, size=12)
        ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=6)

    wb.save(filename)
    print(f"✅ Saved: '{filename}'")
    return filename


# ============================================================
# SAVE
# ============================================================
if SAVE_CV_RESULTS:
    fname = save_cv_to_xlsx(cv_all_results, property_name, featurization_method)

    try:
        files.download(fname)
    except:
        pass